In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')


In [7]:

# === Schema (For structured output)

class SentimentSchema(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Sentiment of the reivew.')

structured_model_sentiment = model.with_structured_output(SentimentSchema)

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review.')
    tone: Literal['angry', 'frustrated', 'disappointed', 'calm'] = Field(description='The emotional tone expressed by the user.')
    urgency: Literal['low', 'medium', 'high'] = Field(description='How urgent or critical the issue is.')

structured_model_diagnosis = model.with_structured_output(DiagnosisSchema)



In [8]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str


In [9]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model_sentiment.invoke(prompt).sentiment
    print('===> sentiment', sentiment)
    # state['sentiment'] = sentiment
    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if(state['sentiment'] == 'positive'):
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state: ReviewState) -> Any:
    prompt = f"""
    Write a warm thank you message in response to this review: \n\n "{state['review']}\"\n
    Also, Kindly ask the user to leave feedback for our website.
    """
    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState) -> Any:
    prompt = f"""
    Diagnose this negative review: \n\n {state['review']}\n "
    "Return issue_type, tone and urgency.
    """
    response = structured_model_diagnosis.invoke(prompt)
    response_dict = response.model_dump()
    return {'diagnosis': response_dict}

def negative_response(state: ReviewState)-> Any:
    diagnosis = state['diagnosis']
    prompt = f"""
    You are a helpful support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'
    Write an empathetic resolution message.
    """
    response = model.invoke(prompt).content
    return {'response': response}
    

In [10]:
graph = StateGraph(ReviewState)
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()
workflow
initialState = {
    'review': 'This software hangs a lot.'
}

workflow.invoke(initialState)


===> sentiment negative


{'review': 'This software hangs a lot.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Performance',
  'tone': 'frustrated',
  'urgency': 'high'},
 'response': "I am so sorry to hear you're experiencing performance issues. I understand how incredibly frustrating this must be, especially when it's impacting your work and you marked this as high urgency.\n\nPlease know that we are treating this with the highest priority and are actively working to resolve it as quickly as possible. We're committed to getting things back to normal for you.\n\nWe'll provide you with an update as soon as we have more information or a resolution. Thank you for your patience and understanding while we address this."}